# Conditional Spatial Shapley & Shapley Interaction Values

**Standalone external-sharing edition.** All functions are embedded below; no companion Python module, original training notebook, local project modules, weights or research data are needed to run the synthetic example.

This notebook keeps the KNN conditional attribution method and the pairwise SHAP interaction convention of the final case2 Vshale notebook (original cells 20, 71, 155). Training, cross-validation, private paths and case-specific research plots have been removed.

Requirements: Python with `numpy` and `matplotlib`; install `torch` only for a PyTorch model, and use a Jupyter environment to open this file. Example shell command: `python -m pip install numpy matplotlib`.

Run cells in order. The default example is synthetic. For your own data, edit the final integration cell. No actual model or dataset is distributed.


## Input / model / output contract

- Each player is one **entire feature map**. Shapley and interaction values are spatial maps on the **model output grid**.
- Target input is `[F,Hin,Win]`; references are `{case_id: [F,Hin,Win]}`. Both must match the model training preprocessing, channel order, padding, resize and saved normalization.
- `predict(batch)` takes `[B,F,Hin,Win]` and returns `[B,Hout,Wout]`. Inference must be deterministic and independent of batch composition. Adapt other output types in your predictor.
- A reference input population is required. One target input and its existing output alone cannot provide conditional expectations. The model must run on the newly constructed coalition inputs.
- Observed output (`truth`) is optional and used only for MAE/plots. Existing prediction (`predicted_output`) is optional and checked against the full-coalition result.
- A state_dict needs its corresponding model architecture. A supplied initialized model or a TorchScript model can be connected through `make_torch_predictor`.

For OOF explanations, use the target fold's trained model, its saved normalization, and only that fold's training references. For a full-data model, use the permitted reference pool excluding the target. Self-exclusion does not make an in-sample model explanation OOF.


## Method and conventions

For nonempty proper coalitions, nearest references are selected using RMSE across selected normalized feature maps and the entire spatial grid. Observed channels remain from the target; missing channels are copied jointly from each selected reference. Their model outputs are averaged. The empty coalition averages predictions for up to 20 randomly sampled eligible references (or all references if `baseline_samples=None`). Full coalition uses the target input directly.

All `2**F` coalitions are enumerated: Shapley summation is exact for these coalition values, while the conditional expectation is an empirical KNN approximation. Default K is 3. Target reference IDs are excluded; callers must also remove duplicate target data under different IDs.

For `S` excluding the relevant indices:

`phi_i = sum_S |S|!(F-|S|-1)!/F! * [v(S+i)-v(S)]`

`I_ij = sum_S |S|!(F-|S|-2)!/[2(F-1)!] * [v(S+i+j)-v(S+i)-v(S+j)+v(S)]`

`I_ii = phi_i - sum_{j != i} I_ij`

The matrix is symmetric. Each unique pair's combined allocation is `2*I_ij`. The diagonal is the main effect. Higher-order effects can contribute to these pairwise allocations; they are not restricted to pure second-order model terms.

Set `output_bounds=(0,1)` to reproduce the original Vshale clipping **after each coalition's prediction average**. Generic regression defaults to unrestricted `None`. Inverse target scaling belongs inside the predictor so it applies to every coalition. Signed contributions are never clipped to the target range.

The refactor uses float64 for distances and accumulation (model inputs remain float32) and a local seeded Python RNG. It removes hidden global configuration. Small numerical differences from original float32 calculations are possible. Reference insertion order resolves equal-distance ties as in the source.


## 1. Standalone calculation, adapters and result I/O

In [ ]:
"""Standalone conditional spatial Shapley and pairwise SHAP interaction values.

Extracted/refactored from the final case2 Vshale fold-CV notebook, cells 20,
71 and 155. No training, local dataset module, or checkpoint is required here.
Each player is an entire input feature map; attribution is evaluated at each
output pixel. All inputs must already be in the model's training input space.
"""
from itertools import combinations
from math import factorial
from pathlib import Path
import json
import random

import numpy as np


def _array(value, name, ndim=None):
    if hasattr(value, 'detach'):
        value = value.detach().cpu().numpy()
    value = np.asarray(value, dtype=np.float64)
    if ndim is not None and value.ndim != ndim:
        raise ValueError(f'{name}: expected {ndim} dimensions, got {value.shape}')
    if value.size == 0 or not np.isfinite(value).all():
        raise ValueError(f'{name}: must be nonempty and finite')
    return value


def normalize_inputs(inputs, mean, std):
    """Apply SAVED training channel statistics to [F,H,W] or [B,F,H,W].

    Call only for raw inputs. Already normalized inputs must not be normalized
    again. Resize/pad/channel ordering must also match the model's training.
    """
    x = _array(inputs, 'inputs')
    if x.ndim not in (3, 4):
        raise ValueError('inputs must be [F,H,W] or [B,F,H,W]')
    f = x.shape[-3]
    mean = _array(mean, 'mean').reshape(-1)
    std = _array(std, 'std').reshape(-1)
    if mean.size != f or std.size != f or np.any(std <= 0):
        raise ValueError('mean/std must have F entries; std must be positive')
    return ((x - mean[:, None, None]) / std[:, None, None]).astype(np.float32)


def make_torch_predictor(model, device='cpu', output_channel=0,
                         output_transform=None):
    """Adapt a supplied torch.nn.Module to predict(batch) -> [B,H,W].

    model must return a tensor [B,C,H,W]. Provide your own wrapper for models
    returning dictionaries/tuples. output_transform is an optional NumPy
    function for inverse target scaling; apply it to EVERY model prediction.
    PyTorch is imported only when this adapter is used.
    """
    import torch
    if not isinstance(output_channel, int) or output_channel < 0:
        raise ValueError('output_channel must be a nonnegative integer')
    model = model.to(device).eval()

    def predict(batch):
        with torch.no_grad():
            x = torch.as_tensor(batch, dtype=torch.float32, device=device)
            y = model(x)
            if not isinstance(y, torch.Tensor) or y.ndim != 4:
                raise ValueError('model must return tensor [B,C,H,W]')
            if output_channel >= y.shape[1]:
                raise ValueError('output_channel is outside the model output')
            y = y[:, output_channel].detach().cpu().numpy()
        return output_transform(y) if output_transform is not None else y

    return predict


def coalitions(n_features):
    """Same size-first enumeration as the source notebook."""
    if not isinstance(n_features, int) or n_features < 1:
        raise ValueError('n_features must be a positive integer')
    return [frozenset(s) for k in range(n_features + 1)
            for s in combinations(range(n_features), k)]


def shapley_from_coalitions(coalition_preds, n_features):
    """Exact Shapley sums and symmetric pairwise interaction matrix.

    v(S) must contain all 2**F coalitions, each with output shape [H,W].
    Off diagonal weight: |S|!(F-|S|-2)! / (2(F-1)!).
    Diagonal: main_i = phi_i - sum_{j != i} interaction_ij.
    A unique pair's combined allocation is 2*interaction_ij; do not omit
    the factor 1/2 in the matrix convention used by the original notebook.
    Returns phi [H,W,F], interaction [H,W,F,F], main [H,W,F].
    """
    subsets = coalitions(n_features)
    if set(coalition_preds) != set(subsets):
        raise ValueError('coalition_preds must contain every subset of 0..F-1')
    cp = {s: _array(coalition_preds[s], f'v({sorted(s)})', 2) for s in subsets}
    shape = cp[frozenset()].shape
    if any(v.shape != shape for v in cp.values()):
        raise ValueError('all coalition output maps must have the same shape')
    f = n_features
    phi = np.zeros((*shape, f), dtype=np.float64)
    interaction = np.zeros((*shape, f, f), dtype=np.float64)
    for i in range(f):
        for s in subsets:
            if i not in s:
                k = len(s)
                weight = factorial(k) * factorial(f-k-1) / factorial(f)
                phi[..., i] += weight * (cp[s | {i}] - cp[s])
    for i, j in combinations(range(f), 2):
        for s in subsets:
            if i not in s and j not in s:
                k = len(s)
                weight = factorial(k) * factorial(f-k-2) / (2*factorial(f-1))
                delta = cp[s | {i, j}] - cp[s | {i}] - cp[s | {j}] + cp[s]
                interaction[..., i, j] += weight * delta
        interaction[..., j, i] = interaction[..., i, j]
    for i in range(f):
        interaction[..., i, i] = phi[..., i] - interaction[..., i, :].sum(-1)
    main = np.diagonal(interaction, axis1=-2, axis2=-1).copy()
    return phi, interaction, main


def explain_conditional(predict, target_input, reference_inputs, *,
                        feature_names, target_id='target', n_neighbors=3,
                        baseline_samples=20, seed=42, batch_size=1,
                        output_shape=None, output_bounds=None, truth=None,
                        predicted_output=None, metadata=None):
    """Compute KNN conditional spatial Shapley and interaction values.

    predict: callable taking [B,F,Hin,Win], returning [B,Hout,Wout].
    target_input: [F,Hin,Win], already normalized/preprocessed.
    reference_inputs: {case_id: input [F,Hin,Win]}, using identical preprocessing.
    feature_names: ordered F names matching model channels.
    target_id: matching reference ID is excluded, even for the baseline.
    n_neighbors: nonempty proper coalition KNN count (source default 3).
    baseline_samples: uniformly sample up to this many references for v(empty)
        (source default 20). None uses all eligible references.
    seed: local Python random seed; does not mutate global RNG state.
    batch_size: inference batch size, default 1 matches source forwards.
    output_shape: optional upper-left crop [H,W], for padded outputs.
    output_bounds: None for unrestricted outputs; (0,1) reproduces Vshale
        clipping AFTER averaging each coalition, as in source cell 20.
    truth: optional observed target [H,W] for visualization/MAE only.
    predicted_output: optional supplied prediction [H,W] in the same response
        space; checked against v(full), NEVER substituted for model inference.
    metadata: optional JSON-compatible provenance (fold/model/reference scope).

    All 2**F coalitions are enumerated exactly; conditional expectations are
    empirical KNN approximations. Selected channels are copied from target,
    missing channels jointly from one reference, preserving their joint maps.
    Deterministic, batch-independent inference is required. No latent sampling.
    """
    x = _array(target_input, 'target_input', 3).astype(np.float32)
    f = x.shape[0]
    names = list(feature_names)
    if len(names) != f or any(not isinstance(n, str) or not n for n in names):
        raise ValueError('feature_names must contain F nonempty strings')
    if len(set(names)) != f:
        raise ValueError('feature_names must be unique')
    if not isinstance(target_id, str):
        raise ValueError('target_id must be a string')
    for name, val in [('n_neighbors', n_neighbors), ('batch_size', batch_size)]:
        if not isinstance(val, int) or val < 1:
            raise ValueError(f'{name} must be a positive integer')
    if baseline_samples is not None and (not isinstance(baseline_samples, int)
                                         or baseline_samples < 1):
        raise ValueError('baseline_samples must be positive or None')
    refs = {}
    for cid, value in reference_inputs.items():
        if not isinstance(cid, str):
            raise ValueError('reference IDs must be strings')
        if cid == target_id:
            continue
        a = _array(value, f'reference {cid}', 3).astype(np.float32)
        if a.shape != x.shape:
            raise ValueError(f'reference {cid}: shape must be {x.shape}')
        refs[cid] = a
    if not refs:
        raise ValueError('no references remain after target self-exclusion')
    if output_shape is not None:
        if (len(output_shape) != 2 or any(not isinstance(v, (int, np.integer))
                                        or v < 1 for v in output_shape)):
            raise ValueError('output_shape must be two positive integers')
        output_shape = tuple(int(v) for v in output_shape)
    if output_bounds is not None:
        bounds = _array(output_bounds, 'output_bounds').reshape(-1)
        if bounds.size != 2 or bounds[0] >= bounds[1]:
            raise ValueError('output_bounds must be (lower, upper), lower < upper')
        output_bounds = tuple(float(v) for v in bounds)
    provenance = dict(metadata or {})
    json.dumps(provenance)  # Validate before running expensive inference.

    def forward(inputs):
        total, count = None, 0
        for start in range(0, len(inputs), batch_size):
            batch = np.stack(inputs[start:start+batch_size])
            y = _array(predict(batch), 'predict(batch)', 3)
            if y.shape[0] != len(batch):
                raise ValueError('predict must preserve the batch dimension')
            if output_shape is not None:
                h, w = output_shape
                if y.shape[1] < h or y.shape[2] < w:
                    raise ValueError('output_shape exceeds model output shape')
                y = y[:, :h, :w]
            if total is None:
                total = np.zeros(y.shape[1:], dtype=np.float64)
            elif total.shape != y.shape[1:]:
                raise ValueError('model output spatial shape changed between batches')
            total += y.sum(axis=0)
            count += len(y)
        mean = total / count
        return np.clip(mean, *output_bounds) if output_bounds is not None else mean

    cp, neighbor_ids, neighbor_distances = {}, {}, {}
    rng = random.Random(seed)
    for s in coalitions(f):
        if len(s) == f:
            ids, distances, inputs = [target_id], [0.0], [x]
        elif not s:
            items = list(refs.items())
            sampled = (items if baseline_samples is None else
                       rng.sample(items, min(baseline_samples, len(items))))
            ids = [cid for cid, _ in sampled]
            inputs = [a for _, a in sampled]
            distances = []
        else:
            selected = sorted(s)
            # Float64 distance accumulation prevents overflow for large maps.
            ranked = []
            for cid, a in refs.items():
                diff = x[selected].astype(np.float64) - a[selected]
                ranked.append((float(np.sqrt(np.mean(diff**2))), cid))
            # Stable tie order matches insertion order in the source cache.
            ranked.sort(key=lambda item: item[0])
            ranked = ranked[:min(n_neighbors, len(ranked))]
            ids = [cid for _, cid in ranked]
            distances = [d for d, _ in ranked]
            missing = [i for i in range(f) if i not in s]
            inputs = []
            for cid in ids:
                hybrid = x.copy()
                hybrid[missing] = refs[cid][missing]
                inputs.append(hybrid)
        cp[s] = forward(inputs)
        neighbor_ids[s], neighbor_distances[s] = ids, distances
    phi, interaction, main = shapley_from_coalitions(cp, f)
    full, base = cp[frozenset(range(f))], cp[frozenset()]
    result = dict(case_id=target_id, feature_names=names, shapley_2d=phi,
                  interact_2d=interaction, main_effect_2d=main,
                  pred_full=full, pred_base=base, coalition_preds=cp,
                  coalition_neighbor_ids=neighbor_ids,
                  coalition_neighbor_distances=neighbor_distances,
                  metadata=provenance,
                  settings=dict(n_neighbors=n_neighbors,
                                baseline_samples=baseline_samples, seed=seed,
                                batch_size=batch_size, output_shape=output_shape,
                                output_bounds=output_bounds))
    for key, value in [('true_2d', truth), ('supplied_prediction', predicted_output)]:
        if value is not None:
            a = _array(value, key, 2)
            if a.shape != full.shape:
                raise ValueError(f'{key}: expected output shape {full.shape}')
            if key == 'supplied_prediction' and not np.allclose(a, full, rtol=1e-5, atol=1e-6):
                raise ValueError('supplied prediction differs from v(full); check '
                                 'normalization, output scaling/crop/clipping or model')
            result[key] = a
    result['diagnostics'] = check_efficiency(result)
    return result


def check_efficiency(result):
    """Report pixelwise efficiency, interaction row sums and symmetry."""
    phi, m = result['shapley_2d'], result['interact_2d']
    delta = result['pred_full'] - result['pred_base']
    diagnostics = {
        'efficiency_max_abs': float(np.abs(phi.sum(-1)-delta).max()),
        'interaction_row_sum_max_abs': float(np.abs(m.sum(-1)-phi).max()),
        'interaction_symmetry_max_abs': float(np.abs(m-m.swapaxes(-1,-2)).max()),
        'interaction_total_max_abs': float(np.abs(m.sum((-1,-2))-delta).max()),
    }
    if 'true_2d' in result:
        diagnostics['prediction_mae'] = float(np.abs(result['pred_full']-result['true_2d']).mean())
    return diagnostics


def save_result(result, path):
    """Save all coalitions, audit IDs, settings and maps; no pickled objects."""
    subsets = coalitions(len(result['feature_names']))
    payload = {k: result[k] for k in ['shapley_2d', 'interact_2d',
               'main_effect_2d', 'pred_full', 'pred_base']}
    for k in ['true_2d', 'supplied_prediction']:
        if k in result:
            payload[k] = result[k]
    payload['coalition_masks'] = np.array([sum(1 << i for i in s) for s in subsets])
    payload['coalition_values'] = np.stack([result['coalition_preds'][s] for s in subsets])
    info = {k: result[k] for k in ['case_id', 'feature_names', 'metadata', 'settings', 'diagnostics']}
    info['neighbor_ids'] = [result['coalition_neighbor_ids'][s] for s in subsets]
    info['neighbor_distances'] = [result['coalition_neighbor_distances'][s] for s in subsets]
    payload['info_json'] = np.asarray(json.dumps(info, ensure_ascii=False))
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('wb') as handle:
        np.savez_compressed(handle, **payload)


def load_result(path):
    """Load files created by save_result with allow_pickle=False."""
    with np.load(path, allow_pickle=False) as data:
        info = json.loads(str(data['info_json'].item()))
        f = len(info['feature_names'])
        subsets = [frozenset(i for i in range(f) if int(mask) & (1 << i))
                   for mask in data['coalition_masks']]
        result = {k: data[k].copy() for k in ['shapley_2d', 'interact_2d',
                  'main_effect_2d', 'pred_full', 'pred_base']}
        for k in ['true_2d', 'supplied_prediction']:
            if k in data:
                result[k] = data[k].copy()
        result.update({k: info[k] for k in ['case_id', 'feature_names', 'metadata', 'settings', 'diagnostics']})
        result['coalition_preds'] = dict(zip(subsets, data['coalition_values'].copy()))
        result['coalition_neighbor_ids'] = dict(zip(subsets, info['neighbor_ids']))
        result['coalition_neighbor_distances'] = dict(zip(subsets, info['neighbor_distances']))
    return result




## 2. Main visualizations

Functions return Matplotlib figures. `path=` optionally saves a PNG. No project-specific AOI modules are required: pass a boolean mask on the output grid to `plot_summary`. Physical axes require an explicit `extent=[xmin,xmax,zmax,zmin]`; otherwise plots show pixel indices.

In [ ]:
# Visualization functions use only Matplotlib; no local AOI modules or SHAP
# library are needed. Every function returns a figure; none writes implicitly.
def _finish(fig, path):
    if path is not None:
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(path, dpi=180, bbox_inches='tight')
    return fig


def _map(ax, values, title, *, signed=True, limit=None, extent=None, bounds=None):
    if signed:
        limit = max(float(np.abs(values).max()) if limit is None else limit, 1e-12)
        vmin, vmax, cmap = -limit, limit, 'RdBu_r'
    else:
        vmin, vmax = bounds if bounds is not None else (None, None)
        cmap = 'viridis'
    image = ax.imshow(values, aspect='auto', origin='upper', cmap=cmap,
                      vmin=vmin, vmax=vmax, extent=extent, interpolation='nearest')
    ax.set_title(title, fontsize=10)
    ax.set_xlabel('X' if extent is not None else 'Column index')
    ax.set_ylabel('Depth' if extent is not None else 'Row index')
    ax.figure.colorbar(image, ax=ax, fraction=0.046, pad=0.04)


def plot_overview(result, *, extent=None, output_bounds=None, path=None):
    """Prediction, optional truth/residual, baseline and efficiency maps.

    extent=[xmin,xmax,zmax,zmin] sets physical axis coordinates. Supply the
    actual physical domain, accounting for any training resize/crop yourself.
    """
    import matplotlib.pyplot as plt
    panels = [(result['pred_base'], 'Reference baseline v(empty)', False),
              (result['pred_full'], 'Model prediction v(full)', False)]
    if 'true_2d' in result:
        panels += [(result['true_2d'], 'Observed output', False),
                   (result['pred_full']-result['true_2d'], 'Prediction - observation', True)]
    delta = result['pred_full']-result['pred_base']
    summed = result['shapley_2d'].sum(-1)
    panels += [(delta, 'Prediction - baseline', True), (summed, 'Sum of Shapley maps', True),
               (summed-delta, 'Efficiency residual', True)]
    cols = 3
    fig, axes = plt.subplots((len(panels)+cols-1)//cols, cols,
                             figsize=(14, 3.7*((len(panels)+cols-1)//cols)), squeeze=False,
                             layout='constrained')
    limit = max(float(np.abs(delta).max()), float(np.abs(summed).max()), 1e-12)
    for ax, (a, title, signed) in zip(axes.flat, panels):
        _map(ax, a, title, signed=signed, extent=extent, bounds=output_bounds,
             limit=limit if title in ('Prediction - baseline', 'Sum of Shapley maps') else None)
    for ax in list(axes.flat)[len(panels):]:
        ax.set_visible(False)
    fig.suptitle(f'Conditional spatial Shapley | {result["case_id"]}')
    return _finish(fig, path)


def plot_feature_maps(result, *, kind='shapley', extent=None, ncols=4, path=None):
    """Signed total/main maps, or absolute attribution shares (%).

    share_i = 100*abs(phi_i)/sum_j abs(phi_j); this is an attribution share,
    not predictive accuracy or explained-variance percentage.
    """
    import matplotlib.pyplot as plt
    values = result['shapley_2d']
    if kind == 'main':
        values = result['main_effect_2d']
    elif kind == 'share':
        denom = np.abs(values).sum(-1, keepdims=True)
        values = np.divide(100*np.abs(values), denom, out=np.zeros_like(values), where=denom > 0)
    elif kind != 'shapley':
        raise ValueError('kind must be shapley, main or share')
    if not isinstance(ncols, int) or ncols < 1:
        raise ValueError('ncols must be positive')
    names = result['feature_names']
    cols = min(ncols, len(names))
    rows = (len(names)+cols-1)//cols
    fig, axes = plt.subplots(rows, cols, figsize=(4*cols, 3.5*rows),
                             squeeze=False, layout='constrained')
    limit = max(float(np.abs(values).max()), 1e-12)
    for i, ax in enumerate(axes.flat):
        if i >= len(names):
            ax.set_visible(False)
            continue
        _map(ax, values[..., i], names[i], signed=kind != 'share', limit=limit,
             extent=extent, bounds=(0,100) if kind == 'share' else None)
    fig.suptitle(f'{kind.capitalize()} maps | {result["case_id"]}')
    return _finish(fig, path)


def plot_interaction_maps(result, *, pairs=None, extent=None, path=None):
    """Unique off-diagonal pair maps, with one shared symmetric color scale."""
    import matplotlib.pyplot as plt
    names = result['feature_names']
    pairs = list(combinations(range(len(names)), 2)) if pairs is None else list(pairs)
    if not pairs or any(i == j or min(i,j) < 0 or max(i,j) >= len(names) for i,j in pairs):
        raise ValueError('supply at least one valid pair of different feature indices')
    cols = min(4, len(pairs))
    rows = (len(pairs)+cols-1)//cols
    fig, axes = plt.subplots(rows, cols, figsize=(4*cols, 3.5*rows),
                             squeeze=False, layout='constrained')
    m = result['interact_2d']
    limit = max(max(float(np.abs(m[..., i,j]).max()) for i,j in pairs), 1e-12)
    for ax, (i,j) in zip(axes.flat, pairs):
        _map(ax, m[..., i,j], f'{names[i]} x {names[j]}', limit=limit, extent=extent)
    for ax in list(axes.flat)[len(pairs):]:
        ax.set_visible(False)
    fig.suptitle(f'Pairwise interaction maps | {result["case_id"]}')
    return _finish(fig, path)


def _region_mask(result, mask):
    shape = result['pred_full'].shape
    if mask is None:
        return np.ones(shape, dtype=bool)
    mask = np.asarray(mask)
    if mask.shape != shape or mask.dtype != bool or not mask.any():
        raise ValueError('AOI mask must be nonempty boolean [Hout,Wout]')
    return mask


def plot_summary(result, *, mask=None, region_name='All pixels', path=None):
    """Feature importance and signed/absolute-mean interaction matrices.

    AOI masks are supplied by the caller on the OUTPUT grid. Diagonal entries
    are main effects, off-diagonal entries are interactions. mean(abs(value))
    differs from abs(mean(value)); signed spatial means can cancel.
    """
    import matplotlib.pyplot as plt
    mask = _region_mask(result, mask)
    names = result['feature_names']
    importance = np.abs(result['shapley_2d'][mask]).mean(0)
    order = np.argsort(importance)
    signed = result['interact_2d'][mask].mean(0)
    strength = np.abs(result['interact_2d'][mask]).mean(0)
    fig, axes = plt.subplots(1, 3, figsize=(17, max(4, len(names)*0.6)), layout='constrained')
    axes[0].barh(np.arange(len(names)), importance[order])
    axes[0].set_yticks(np.arange(len(names)), [names[i] for i in order])
    axes[0].set_xlabel('Mean absolute Shapley value')
    axes[0].set_title('Feature importance')
    for ax, a, title, cmap in [(axes[1], signed, 'Signed mean interaction', 'RdBu_r'),
                               (axes[2], strength, 'Mean absolute interaction', 'magma')]:
        limit = max(float(np.abs(a).max()), 1e-12)
        im = ax.imshow(a, cmap=cmap, vmin=-limit if cmap == 'RdBu_r' else 0, vmax=limit)
        ax.set_xticks(range(len(names)), names, rotation=45, ha='right')
        ax.set_yticks(range(len(names)), names)
        ax.set_title(title + '\nDiagonal: main effect')
        for i in range(len(names)):
            for j in range(len(names)):
                ax.text(j, i, f'{a[i,j]:.2g}', ha='center', va='center', fontsize=8,
                        color='white' if (abs(a[i,j])/limit > 0.6 if cmap == 'RdBu_r' else a[i,j]/limit < 0.5) else 'black')
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    fig.suptitle(f'{region_name} | {result["case_id"]}')
    return _finish(fig, path)


def plot_dominance(result, *, extent=None, path=None):
    """Dominant absolute-contribution feature and its share. Zero-total
    pixels are masked, because they have no dominant feature. Ties select
    the first channel. Shares do not retain the sign of contributions.
    """
    import matplotlib.pyplot as plt
    from matplotlib.colors import ListedColormap, BoundaryNorm
    names = result['feature_names']
    a = np.abs(result['shapley_2d'])
    total = a.sum(-1)
    dominant = np.ma.array(a.argmax(-1), mask=total == 0)
    share = np.divide(100*a.max(-1), total, out=np.zeros_like(total), where=total > 0)
    cmap = ListedColormap(plt.get_cmap('tab20')(np.linspace(0, 1, len(names))))
    norm = BoundaryNorm(np.arange(len(names)+1)-0.5, len(names))
    fig, axes = plt.subplots(1, 2, figsize=(12, 4), layout='constrained')
    im = axes[0].imshow(dominant, cmap=cmap, norm=norm, aspect='auto', extent=extent)
    cb = fig.colorbar(im, ax=axes[0], ticks=range(len(names)))
    cb.ax.set_yticklabels(names)
    axes[0].set_title('Dominant feature by absolute Shapley value')
    _map(axes[1], share, 'Dominant attribution share (%)', signed=False,
         bounds=(0,100), extent=extent)
    axes[0].set_xlabel('X' if extent is not None else 'Column index')
    axes[0].set_ylabel('Depth' if extent is not None else 'Row index')
    fig.suptitle(result['case_id'])
    return _finish(fig, path)


def plot_dependencies(result, feature_values, *, mask=None, max_pixels=3000,
                      seed=42, path=None):
    """Feature-value vs Shapley scatter on matched output pixels.

    feature_values must be [Hout,Wout,F] aligned to the output grid. Can be
    physical feature values or normalized values (label your units). The
    function does not silently resize or equate input and output coordinates.
    """
    import matplotlib.pyplot as plt
    fv = _array(feature_values, 'feature_values', 3)
    if fv.shape != result['shapley_2d'].shape:
        raise ValueError('feature_values must be aligned [Hout,Wout,F]')
    if not isinstance(max_pixels, int) or max_pixels < 1:
        raise ValueError('max_pixels must be positive')
    mask = _region_mask(result, mask)
    values, phi = fv[mask], result['shapley_2d'][mask]
    idx = np.random.default_rng(seed).choice(len(values), min(max_pixels,len(values)), replace=False)
    names = result['feature_names']
    cols = min(3, len(names)); rows = (len(names)+cols-1)//cols
    fig, axes = plt.subplots(rows, cols, figsize=(4.5*cols,3.5*rows), squeeze=False, layout='constrained')
    for i, ax in enumerate(axes.flat):
        if i >= len(names):
            ax.set_visible(False)
            continue
        ax.scatter(values[idx,i], phi[idx,i], s=7, alpha=0.3, rasterized=True)
        ax.axhline(0, color='gray', linewidth=0.7)
        ax.set_xlabel(names[i] + ' input value')
        ax.set_ylabel('Shapley contribution')
        ax.set_title(names[i])
    fig.suptitle(f'Feature dependencies | {result["case_id"]}')
    return _finish(fig, path)


## 3. Runnable synthetic example

This small example demonstrates the complete interface with three features and a deterministic NumPy model. It is not research data and does not claim predictive performance. Replace the model and inputs in section 5 for real analysis.


In [ ]:
import matplotlib.pyplot as plt

rng = np.random.default_rng(21)
FEATURE_NAMES = ['Feature A', 'Feature B', 'Feature C']
h, w = 24, 32
z, horizontal = np.mgrid[0:1:complex(h), 0:1:complex(w)]
target_input = np.stack([horizontal, z, np.sin(3*horizontal)*np.cos(3*z)]).astype(np.float32)
reference_inputs = {
    f'reference_{i:03d}': np.stack([
        horizontal + rng.normal(0,0.18,(h,w)),
        z + rng.normal(0,0.18,(h,w)),
        rng.normal(0,0.3,(h,w)),
    ]).astype(np.float32)
    for i in range(24)
}

def demo_predict(batch):
    # Input [B,F,H,W] -> output [B,H,W]. Includes a pairwise term.
    y = 0.2 + 0.25*batch[:,0] + 0.2*batch[:,1] + 0.15*batch[:,2]
    return y + 0.15*batch[:,0]*batch[:,1]

observed_output = demo_predict(target_input[None])[0] + rng.normal(0,0.01,(h,w))
result = explain_conditional(
    demo_predict, target_input, reference_inputs,
    feature_names=FEATURE_NAMES, target_id='synthetic_target',
    n_neighbors=3, baseline_samples=20, seed=42, batch_size=1,
    output_bounds=(0,1), truth=observed_output,
    predicted_output=np.clip(demo_predict(target_input[None])[0],0,1),
    metadata={'model_scope':'synthetic demonstration', 'reference_scope':'24 synthetic maps'},
)
print('Shapley:', result['shapley_2d'].shape)
print('Interaction:', result['interact_2d'].shape)
print(json.dumps(result['diagnostics'], indent=2))
assert result['diagnostics']['efficiency_max_abs'] < 1e-8
assert result['diagnostics']['interaction_row_sum_max_abs'] < 1e-8


In [ ]:
plot_overview(result, output_bounds=(0,1))
plt.show()
plot_feature_maps(result, kind='shapley')
plt.show()
plot_feature_maps(result, kind='main')
plt.show()
plot_interaction_maps(result)
plt.show()
plot_summary(result)
plt.show()
plot_dominance(result)
plt.show()
plot_feature_maps(result, kind='share')
plt.show()
# Demo input and output grids match. For other grids, align explicitly.
plot_dependencies(result, target_input.transpose(1,2,0))
plt.show()
# An arbitrary output-grid AOI example (no geology-specific thresholds).
aoi_mask = np.zeros(result['pred_full'].shape, dtype=bool)
aoi_mask[h//2:] = True
plot_summary(result, mask=aoi_mask, region_name='Lower half of output grid')
plt.show()


## 4. Save/reload results

The new NPZ format includes all coalition maps, interaction tensors, neighbor IDs/distances, settings and metadata. It requires no pickle and is separate from the original notebook's object-array NPZ format. Save each case separately for larger studies.

`sum_i phi_i = v(full)-v(empty)` at each output pixel. Interaction row sums reconstruct `phi_i`; all matrix entries together reconstruct the prediction-baseline difference. `diagnostics` measures these identities. It does not measure the accuracy of the KNN conditional approximation.

Feature importance is `mean(abs(phi))`. Signed spatial means may cancel. Absolute attribution shares are not accuracy or variance-explained percentages.


In [ ]:
OUTPUT_DIR = Path('spatial_shapley_outputs')
save_result(result, OUTPUT_DIR / 'synthetic_target.npz')
restored = load_result(OUTPUT_DIR / 'synthetic_target.npz')
np.testing.assert_allclose(restored['shapley_2d'], result['shapley_2d'])
plot_summary(restored, path=OUTPUT_DIR / 'synthetic_interaction_summary.png')
plt.show()
print('Saved under:', OUTPUT_DIR.resolve())


## 5. External-data integration (edit before enabling)

Prepare these files relative to your own working directory:

- `input_target.npy`: float array `[F,Hin,Win]`.
- `reference_inputs.npz`: each key is a unique case ID with `[F,Hin,Win]` float array. Use only the references allowed for this target/model.
- `normalization.npz`: `mean` and `std`, each `[F]`, copied from training, only if inputs are raw. Do not recompute them here.
- `model_scripted.pt`: your deterministic TorchScript model returning `[B,C,Hout,Wout]`, or replace the loading lines with your already initialized `torch.nn.Module`.
- `output_observed.npy`: optional observed `[Hout,Wout]`, already in response units.
- `output_predicted.npy`: optional prior prediction `[Hout,Wout]`, matching response scaling, crop and clipping.

The below example uses the original seven input feature names, assumes single-channel Vshale output in response units, and disables extra preprocessing by default because inputs are assumed already normalized. Change these settings to your model's contract. For a checkpoint state_dict, instantiate your architecture and load its generator weights first, then pass that model to the adapter. This notebook intentionally does not prescribe the architecture of an external company's model.

If the model output is padded, set `OUTPUT_SHAPE` for an upper-left crop. If the input/output grids differ, do not call `plot_dependencies` until feature values have been explicitly aligned to the output grid.


In [ ]:
RUN_EXTERNAL = False
DATA_DIR = Path('your_analysis_data')
EXTERNAL_OUTPUT_DIR = Path('external_shapley_results')
FEATURE_NAMES_EXTERNAL = ['AI', 'RAI', 'Envelope', 'PBI', 'SD_Low', 'SD_High', 'Amplitude']
TARGET_ID = 'your_target_case_id'
INPUTS_ALREADY_NORMALIZED = True
OUTPUT_SHAPE = None
OUTPUT_BOUNDS = (0,1)  # Original Vshale. Use None for unrestricted regression.
DEVICE = 'cpu'

if RUN_EXTERNAL:
    import torch
    external_x = np.load(DATA_DIR / 'input_target.npy', allow_pickle=False)
    with np.load(DATA_DIR / 'reference_inputs.npz', allow_pickle=False) as ref_file:
        external_refs = {cid: ref_file[cid].copy() for cid in ref_file.files}
    if not INPUTS_ALREADY_NORMALIZED:
        with np.load(DATA_DIR / 'normalization.npz', allow_pickle=False) as norm:
            external_x = normalize_inputs(external_x, norm['mean'], norm['std'])
            external_refs = {cid: normalize_inputs(x, norm['mean'], norm['std'])
                             for cid,x in external_refs.items()}
    # Replace with your existing model if you use a different checkpoint format.
    model = torch.jit.load(str(DATA_DIR / 'model_scripted.pt'), map_location=DEVICE)
    predict = make_torch_predictor(model, device=DEVICE, output_channel=0)
    # Optional example inverse scaling: output_transform=lambda y: y*saved_scale+saved_offset
    truth_path = DATA_DIR / 'output_observed.npy'
    prior_path = DATA_DIR / 'output_predicted.npy'
    external_result = explain_conditional(
        predict, external_x, external_refs,
        feature_names=FEATURE_NAMES_EXTERNAL, target_id=TARGET_ID,
        n_neighbors=3, baseline_samples=20, seed=42, batch_size=1,
        output_shape=OUTPUT_SHAPE, output_bounds=OUTPUT_BOUNDS,
        truth=np.load(truth_path,allow_pickle=False) if truth_path.exists() else None,
        predicted_output=np.load(prior_path,allow_pickle=False) if prior_path.exists() else None,
        metadata={'model_scope':'EDIT: model training scope',
                  'reference_scope':'EDIT: eligible training/reference population'},
    )
    save_result(external_result, EXTERNAL_OUTPUT_DIR / 'result.npz')
    print(json.dumps(external_result['diagnostics'],indent=2))
    plot_overview(external_result, output_bounds=OUTPUT_BOUNDS,
                  path=EXTERNAL_OUTPUT_DIR / 'overview.png')
    plot_feature_maps(external_result, path=EXTERNAL_OUTPUT_DIR / 'shapley_maps.png')
    plot_feature_maps(external_result, kind='main', path=EXTERNAL_OUTPUT_DIR / 'main_effect_maps.png')
    if len(FEATURE_NAMES_EXTERNAL) > 1:
        plot_interaction_maps(external_result, path=EXTERNAL_OUTPUT_DIR / 'interaction_maps.png')
    plot_summary(external_result, path=EXTERNAL_OUTPUT_DIR / 'interaction_summary.png')
    plot_dominance(external_result, path=EXTERNAL_OUTPUT_DIR / 'dominance.png')
    plt.show()
else:
    print('External integration is disabled. Edit the configuration, then set RUN_EXTERNAL=True.')
